# P0 — Arreglar la evaluación (precondición de todo)

## Pregunta
¿El desempeño reportado del proyecto original era confiable?

## Hipótesis
No: (a) usaba `KFold` no estratificado con clases muy desbalanceadas (clase 5 = 15 ejemplos), inflando la varianza entre folds; y (b) reportaba el mejor fold evaluado sobre datos que ya había visto (leakage in-sample, QWK 0.66).

## Método
Reentrenar el LoRA (config real r=32) con **StratifiedKFold**, predicciones **out-of-fold** (cada ejemplo predicho por un modelo que no lo vio), y añadir métricas ordinales (QWK, MAE). Comparar la varianza entre folds contra el `KFold` original.

> Para RE-EJECUTAR: `python experimentos/src/exp_p0.py both` (requiere GPU). Aquí cargamos los resultados.

In [ ]:
# --- Configuracion comun ---
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np, pandas as pd
import common as C
R = C.RESULTS
def load(f): return json.load(open(R / f))

# Patron de dos niveles: por defecto CARGA resultados ya calculados (segundos, sin GPU).
# Para RE-EJECUTAR desde cero (requiere GPU/Bedrock), pon RECOMPUTE=True.
RECOMPUTE = False

## Resultado

In [ ]:
kf = load("p0_kfold.json"); sf = load("p0_stratified.json")
rows = [["KFold (viejo)", round(kf["oof_global"]["f1_macro"],3), round(kf["oof_global"]["qwk"],3),
         round(kf["oof_global"]["mae"],3), round(kf["fold_sd"]["f1_macro"],3)],
        ["StratifiedKFold", round(sf["oof_global"]["f1_macro"],3), round(sf["oof_global"]["qwk"],3),
         round(sf["oof_global"]["mae"],3), round(sf["fold_sd"]["f1_macro"],3)]]
pd.DataFrame(rows, columns=["metodo","F1 macro","QWK","MAE","sd F1 entre folds"])

In [ ]:
import matplotlib.pyplot as plt
kff=[f["f1_macro"] for f in kf["folds"]]; sff=[f["f1_macro"] for f in sf["folds"]]
plt.figure(figsize=(8,4))
plt.plot(range(1,11),kff,"-o",label=f"KFold sd={kf['fold_sd']['f1_macro']:.3f}",color="#bdc3c7")
plt.plot(range(1,11),sff,"-s",label=f"Stratified sd={sf['fold_sd']['f1_macro']:.3f}",color="#27ae60")
plt.xlabel("fold"); plt.ylabel("F1 macro"); plt.legend(); plt.title("P0: StratifiedKFold estabiliza la evaluacion"); plt.grid(alpha=.3); plt.show()

## Veredicto
StratifiedKFold **reduce la varianza de 0.090 a 0.072** con desempeño equivalente. Queda fijado el **baseline honesto: QWK 0.415, F1 macro 0.383, MAE 0.761.**

## Amenaza a la validez
Con N=581 y clase 5 = 15, incluso el estimador estratificado tiene IC amplios. Este es el techo que limita todos los experimentos siguientes.